# Module 3: The agent loop, stop conditions and turn control

Now that tool calls work, the agent calls the same tool on every turn until it hits the turn limit. That limit is `max_turns` in [`config.toml`](../agent/config.toml), the agent's settings file: the most model calls one run may make. The trace shows the pattern and the loop's code shows why. You apply the supplied fix and check that the run stops on its own.

In [ ]:
# Load the lab's notebook helpers as `lab`, and work in the agent's folder from here on.
import labnb as lab
%cd ../agent

## Start from module 3

This cell runs [`reset.py`](../agent/reset.py), which puts the agent in the state module 3 starts from. It copies any edits you made to `agent/.reset-backup/` first, so nothing you wrote is lost.

In [ ]:
# Put the agent in module 3's starting state: pointed at endpoint B, as you left it in module 2.
!python reset.py 3

The reset leaves the agent on endpoint B, the one you pointed it at in module 2, so tool calls now fire.

## 1. Watch it loop

The lab's question `m3`, from [`questions.py`](../agent/questions.py), asks whether a replacement coolant pump for a K-400 can reach the Denver depot this week. Answering it takes two tool calls: `find_part` to get the pump's part number, then `check_stock` with that part number.

In [ ]:
# Ask the agent the lab's question 'm3':
#   "Can we get a replacement coolant pump for a K-400 to the Denver depot this week?"
!python agent.py --q m3
# Show a table of this notebook's runs, one row each:
#   turns: how many times the agent called the model
#   ended by: what stopped the run
#   tools run: the tools the agent ran, in order
#   answer: what the agent printed
lab.evidence(columns=("turns", "ended by", "tools run", "answer"))

Note how many turns the run took and how it ended. How many tool calls did it make, and did `check_stock` ever run?

## 2. Find what went wrong, from the evidence

The next cell links to this run's trace in MLflow and lists every tool call the run made. Open the trace, count the `model_call_N` spans, and read the tool spans in order.

In [ ]:
# A link to the newest run's trace in MLflow.
lab.trace_link()
# Show each tool the newest run called, with its arguments and exactly what it returned.
lab.tool_results()

Each `find_part` call got the same correct result: the pump `KI-CP-40817` and its seal kit `KI-SK-40233`. The tool works. The model asked for it with the same arguments on every turn until `max_turns` stopped the run at 6. The model isn't repeating itself for no reason. Something is wrong with what the loop gives back to it.

The model keeps no memory between calls. It knows only what's in each request, and the agent sends the conversation so far as the request's `messages` list. So every request has to carry everything the model needs, tool results included. In the trace, select `model_call_1`, then `model_call_2`, and compare their Inputs. How many messages does each one hold? Is the `find_part` result in the second one?

The saved payloads show the same thing without leaving the notebook. [`payloads.py`](../agent/payloads.py) prints the digest you read in module 1, one block per model call. Look at the `request` line in each block: it says how many messages the agent sent.

In [ ]:
# Print a digest of each model call in the newest run: what was sent, and what came back.
!python payloads.py

[`agent.py`](../agent/agent.py) holds the agent's loop. Its `run_agent` function sends `messages` to the model, runs the tools the model asks for and decides when the run is over. The next cell shows it.

In [ ]:
# Show the run_agent function from agent.py, with its line numbers.
lab.show_code("agent.py", "run_agent")

**Your read:** read `run_agent` in [`agent.py`](../agent/agent.py) and find two defects: one in what happens to a tool's result, and one in how the loop decides to stop.

<details><summary>Check your answer</summary>

1. Neither the model's reply nor the tool result is added to `messages`, so every request carries only the system prompt and the question. The model never sees the result and asks for `find_part` again. The loop does record each result: `runlog.add_tool_call` saves it with the run, and the trace shows it in the tool's span. Those records are for you to read. Neither sends anything to the model.
2. The loop decides it's done by checking whether the reply has tool calls, and whether its text contains the words "final answer". Nothing tells the model to write those words. The loop never reads `finish_reason`, the field where the model says why it stopped. If a reply ran out of tokens and was cut off (`finish_reason` is `length`), this loop would hand back the half-finished text as the answer.

Defect 1 causes the loop you saw. Defect 2 cost nothing in this run, because every reply asked for a tool. It shows up as soon as a reply is cut off.
</details>

## 3. Fix the loop

The fixed loop is supplied in [`solutions/module-03/agent.py`](../agent/solutions/module-03/agent.py), a corrected copy of `agent.py`. The next cell copies it over `agent.py`, then shows what changed: lines starting with `-` were removed and lines starting with `+` were added.

In [ ]:
# Copy the supplied fix over agent.py.
!cp solutions/module-03/agent.py .
# Show how many lines changed in agent.py, then the changes themselves (the first 80 lines).
!git diff --stat -- agent.py
!git diff -- agent.py | head -80

Two things now go back into `messages` on every turn. First comes the model's own reply, which says which tools it asked for. The new `assistant_message` function turns that reply into a message. Then comes one `role: tool` message per result. The model needs both to match each result to its request.

Every tool call the model makes has an `id`. The agent sends each result back with that `id` as its `tool_call_id`, so the model knows which request the result answers. That matters here because the model asks for two `check_stock` calls at once.

The loop also branches on `finish_reason` now. It carries on when the value is `tool_calls` and returns the answer when it's `stop`. Any other value, such as `length` (the reply was cut off), ends the run and is reported as it is. The turn limit stays, as a safety net.

```mermaid
flowchart TD
  Q["The question"] --> C["Call the model"]
  C --> K["Add the model's reply to messages"]
  K --> D{"finish_reason?"}
  D -- "tool_calls" --> R["Run each tool it asked for"]
  R --> AP["Add each result to messages, with the tool_call_id it answers"]
  AP --> C
  D -- "stop" --> ANS["Return the model's answer"]
  D -- "anything else, such as length" --> OTHER["End the run and report finish_reason as it is"]
  C -. "safety net" .-> CAP["Turn limit reached: stop and say so"]
```

## 4. Confirm it stops

The next cell asks question `m3` again, now with the fixed loop, and lists every tool call the run made.

In [ ]:
# Ask the agent the lab's question 'm3':
#   "Can we get a replacement coolant pump for a K-400 to the Denver depot this week?"
!python agent.py --q m3
# Show a table of this notebook's runs, one row each:
#   turns: how many times the agent called the model
#   ended by: what stopped the run
#   tools run: the tools the agent ran, in order
#   answer: what the agent printed
lab.evidence(columns=("turns", "ended by", "tools run", "answer"))
# A link to the newest run's trace in MLflow.
lab.trace_link()
# Show each tool the newest run called, with its arguments and exactly what it returned.
lab.tool_results()

**This is the module's success signal:** in the last row of the table, `ended by` is `stop`, not `max_turns`, and `turns` is 3: one model call to ask for `find_part`, one to ask for both `check_stock` calls and one to write the answer. Both `check_stock` calls use part numbers from the `find_part` result, and the pump `KI-CP-40817` shows 3 in stock in Denver. In the trace, select `model_call_3`. Its Inputs now hold more than the two messages you saw before: the model's reply asking for `find_part`, a `role: tool` message with its result, the reply asking for both `check_stock` calls, and two more `role: tool` messages.

## If the result differs

If the fixed run still ends on `max_turns`, check these in order:

1. Look at the output of the step 3 cell. The `git diff --stat` line should list `agent.py` with lines changed. If it lists nothing, the copy didn't happen. Run that cell again, then the step 4 cell.
2. If you ran the reset cell again after step 3, it put the broken loop back. Run the step 3 cell, then the step 4 cell.
3. In the newest run's trace, select `model_call_2`. Its Inputs should hold the model's reply asking for `find_part`, with an `id`, followed by a `role: tool` message whose `tool_call_id` is that same `id`. If the `role: tool` message is missing, the fix isn't in `agent.py`.

If the run stopped on its own, check the stock calls. The Denver check must use a part number that `find_part` returned, and the pump `KI-CP-40817` should show 3 in stock. A second `check_stock` call, for the seal kit `KI-SK-40233`, is expected.

If the run ended on `length`, or the model endpoint timed out, the problem is the endpoint, not your fix. Run the step 4 cell again.